# D76 test 多口径评估(单独保存, 不覆盖任何旧文件)

四个口径: full21 / 去11-26(数据异常日, 96%重入) / 去11-26+11-24(病理案例) / 去11-24+12-02+12-03(用户口径)。每口径 RL/fcfs/信号员同日配对。**展示纪律**: 排除必须在页脚注明理由; 11-26 的排除理由最正当(数据异常), 11-24 属真实失败建议以「另见案例剖析」口径处理。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 评估(约 10–20 分钟)

In [ ]:
import json, torch
from derbysim.learn.features import Featurizer
from derbysim.learn.panel import PanelAdapter
from derbysim.learn.panel_models import PanelOptionPolicy
from derbysim.learn.ppo import RunningMoments, TorchPanelDriver
from derbysim.learn.baselines import lookup_table, make_fcfs_policy
from derbysim.learn.rollout import as_factory, evaluate_policy, signaller_replay_factory
from derbysim.learn.scenarios import build_day_trains, day_horizon, load_assets, operating_days
from derbysim.sim.reward import RewardedDynamicEnv
net,params,dwell,ptc=load_assets()
st=torch.load('artifacts/reports/ppo_d73_AB_s0_best.pt',map_location='cpu',weights_only=False)
ck=st['args']; ad=PanelAdapter(net,Featurizer(net,ptc,fv=int(ck.get('feature_version',2))))
pol=PanelOptionPolicy(ck.get('encoder','gnn'),ad.dec_dim,ad.conf_dim,ad.cand_dim,ad.cancel_dim,ad.train_dim,ad.global_dim,hidden=int(ck.get('hidden',64)))
pol.load_state_dict(st['model']); pol.eval()
rn=RunningMoments();
if st.get('ret_norm'): rn.load_state_dict(st['ret_norm'])
envf=lambda: RewardedDynamicEnv(net,params,ptc,dwell,proactive=True,reserve_window='wide',control_classes={'1','2'})
top=lookup_table()
subsets={'full21':[], 'excl_1126':['2023-11-26'], 'excl_1126_1124':['2023-11-26','2023-11-24'],
         'excl_user3':['2023-11-24','2023-12-02','2023-12-03']}
out={}
for name,drop in subsets.items():
    eps=[(str(w['start'])[:16],(lambda w=w: build_day_trains(w)),day_horizon(w))
         for w in operating_days('test') if str(w['start'])[:10] not in drop]
    row={}
    for tag,fac in [('RL',as_factory(TorchPanelDriver(pol,torch.device('cpu'),greedy=True,ret_norm=rn))),
                    ('fcfs',make_fcfs_policy(top)),('signaller',signaller_replay_factory)]:
        b=evaluate_policy(envf,ad,fac,eps); b.pop('eval_episodes',None); row[tag]=b
        print(f"{name:<16}{tag:<10} 完成 {b['eval_completion_rate']:.4f} | 延误 {b['eval_weighted_secondary_delay_s']:>11.0f} | reward {b['eval_episode_reward']:>9.1f}")
    out[name]=row
json.dump(out,open('artifacts/reports/test_subsets.json','w'))
print('saved -> artifacts/reports/test_subsets.json')
